In [1]:
from __future__ import annotations

from dataclasses import dataclass, field
from typing import Dict, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
from scipy import stats
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import log_loss
from sklearn.model_selection import StratifiedKFold

In [2]:
def _make_synthetic(n: int = 40_000, seed: int = 0) -> pd.DataFrame:
    """
    Synthetic bidding log with a KNOWN generating process:

        P(win | bid, a, b) = sigmoid( (bid - threshold(a) - 2*b) / 0.9 )

    so the correct contextualization is {a, b}; c..g are pure noise.
    The demo should (i) pick {a, b}, (ii) reject over-fine subsets via the
    CI-width gate, and (iii) stop adding noise features via CV log-loss.
    """
    rng = np.random.default_rng(seed)
    a = rng.choice(["seg_low", "seg_mid", "seg_high"], size=n, p=[.5, .3, .2])
    b = rng.uniform(0.0, 1.0, size=n)                 # continuous, real signal
    df = pd.DataFrame({
        "a": a,
        "b": b,
        "c": rng.choice(list("WXYZ"), size=n),        # noise, categorical
        "d": rng.normal(size=n),                      # noise, continuous
        "e": rng.integers(0, 5, size=n),              # noise, small-int
        "f": rng.choice(["p", "q"], size=n),          # noise, binary
        "g": rng.exponential(1.0, size=n),            # noise, continuous
        "bid": rng.uniform(0.0, 10.0, size=n),
    })
    threshold = df["a"].map({"seg_low": 2.5, "seg_mid": 4.5,
                             "seg_high": 6.5}).to_numpy()
    p_win = 1.0 / (1.0 + np.exp(-(df["bid"].to_numpy()
                                  - threshold - 2.0 * b) / 0.9))
    df["bid_won"] = rng.binomial(1, p_win)
    return df

In [3]:
df = _make_synthetic()
print(df.shape)
df.head(20)

(40000, 9)


,a,b,c,d,e,f,g,bid,bid_won
0,seg_mid,0.053305,Z,-0.844828,2,p,0.423996,0.928160,0
1,seg_low,0.305871,W,0.751866,2,p,2.182965,9.766868,1
2,seg_low,0.377392,Z,0.270464,2,q,0.536391,4.572124,1
3,seg_low,0.053784,Y,-0.048528,2,q,1.722590,9.760641,1
4,seg_high,0.118610,X,0.838594,1,p,0.061632,0.169339,0
5,seg_high,0.646154,X,0.665922,2,p,2.385558,5.878465,0
6,seg_mid,0.434287,Z,-0.594980,3,p,0.114022,0.302417,0
7,seg_mid,0.724774,Y,0.134895,0,q,0.069943,1.649546,0
8,seg_mid,0.245147,W,-2.257145,3,p,0.790247,2.051623,0
9,seg_high,0.338088,X,-0.931416,3,p,0.625083,9.466259,1


### discretize features

In [4]:
candidate_features: Tuple[str, ...] = ("a", "b", "c", "d", "e", "f", "g")
n_feature_bins: int = 5
n_bid_buckets = 10
ci_method: str = "wilson"
ci_alpha: float = 0.05
max_median_ci_width: float = 0.3
n_folds: int = 5
random_state: int = 42
min_context_train_rows: int = 200

In [5]:
binned = pd.DataFrame(index=df.index)
for col in candidate_features:
    s = df[col]
    if pd.api.types.is_numeric_dtype(s) and s.nunique() > n_feature_bins:
        # duplicates="drop" tolerates heavy ties (fewer bins than asked).
        binned[col] = pd.qcut(s, q=n_feature_bins, duplicates="drop").astype(str)
    else:
        binned[col] = s.astype(str)
print(binned.shape)
binned.head(20)

(40000, 7)


,a,b,c,d,e,f,g
0,seg_mid,"(-0.00099651, 0.199]",Z,"(-0.85, -0.253]",2,p,"(0.221, 0.515]"
1,seg_low,"(0.199, 0.397]",W,"(0.256, 0.836]",2,p,"(1.619, 10.415]"
2,seg_low,"(0.199, 0.397]",Z,"(0.256, 0.836]",2,q,"(0.515, 0.922]"
3,seg_low,"(-0.00099651, 0.199]",Y,"(-0.253, 0.256]",2,q,"(1.619, 10.415]"
4,seg_high,"(-0.00099651, 0.199]",X,"(0.836, 4.267]",1,p,"(-0.0009864000000000001, 0.221]"
5,seg_high,"(0.597, 0.797]",X,"(0.256, 0.836]",2,p,"(1.619, 10.415]"
6,seg_mid,"(0.397, 0.597]",Z,"(-0.85, -0.253]",3,p,"(-0.0009864000000000001, 0.221]"
7,seg_mid,"(0.597, 0.797]",Y,"(-0.253, 0.256]",0,q,"(-0.0009864000000000001, 0.221]"
8,seg_mid,"(0.199, 0.397]",W,"(-4.4110000000000005, -0.85]",3,p,"(0.515, 0.922]"
9,seg_high,"(0.199, 0.397]",X,"(-4.4110000000000005, -0.85]",3,p,"(0.515, 0.922]"


In [6]:
bid_buckets = pd.qcut(df['bid'], q=n_bid_buckets, duplicates="drop")
print(bid_buckets.shape)
bid_buckets.head(20)

(40000,)


0     (-0.0009595000000000001, 0.996]
1                       (9.014, 10.0]
2                       (4.02, 4.989]
3                       (9.014, 10.0]
4     (-0.0009595000000000001, 0.996]
5                      (4.989, 6.014]
6     (-0.0009595000000000001, 0.996]
7                      (0.996, 1.978]
8                      (1.978, 3.003]
9                       (9.014, 10.0]
10                      (7.01, 8.024]
11    (-0.0009595000000000001, 0.996]
12                      (3.003, 4.02]
13                      (4.02, 4.989]
14                      (4.02, 4.989]
15                     (1.978, 3.003]
16                      (6.014, 7.01]
17                      (3.003, 4.02]
18                     (0.996, 1.978]
19                     (0.996, 1.978]
Name: bid, dtype: category
Categories (10, interval[float64, right]): [(-0.0009595000000000001, 0.996] < (0.996, 1.978] < (1.978, 3.003] < (3.003, 4.02] ... (6.014, 7.01] < (7.01, 8.024] < (8.024, 9.014] < (9.014, 10.0]]

In [7]:
y = df['bid_won']
y.shape

(40000,)

In [8]:
def wilson_interval(wins: np.ndarray, n: np.ndarray, alpha: float = 0.05) -> Tuple[np.ndarray, np.ndarray]:
    wins = np.asarray(wins, dtype=float)
    n = np.asarray(n, dtype=float)
    z = stats.norm.ppf(1.0 - alpha / 2.0)          # e.g. 1.96 for alpha=0.05

    with np.errstate(divide="ignore", invalid="ignore"):
        p_hat = np.where(n > 0, wins / n, np.nan)  # empirical win rate
        denom = 1.0 + z**2 / n
        centre = (p_hat + z**2 / (2.0 * n)) / denom
        half = (z * np.sqrt(p_hat * (1.0 - p_hat) / n
                            + z**2 / (4.0 * n**2))) / denom

    lo = np.clip(centre - half, 0.0, 1.0)
    hi = np.clip(centre + half, 0.0, 1.0)
    # Empty cells: no information at all -> [0, 1], i.e. width 1.
    lo = np.where(n > 0, lo, 0.0)
    hi = np.where(n > 0, hi, 1.0)
    return lo, hi


def clopper_pearson_interval(wins: np.ndarray, n: np.ndarray, alpha: float = 0.05) -> Tuple[np.ndarray, np.ndarray]:
    wins = np.asarray(wins, dtype=float)
    n = np.asarray(n, dtype=float)

    with np.errstate(invalid="ignore"):
        # Standard CP construction; the np.where guards handle the edge cases
        # k == 0 (lower bound is exactly 0) and k == n (upper bound exactly 1),
        # where the Beta quantile would be undefined.
        lo = np.where(wins > 0,
                      stats.beta.ppf(alpha / 2.0, wins, n - wins + 1.0), 0.0)
        hi = np.where(wins < n,
                      stats.beta.ppf(1.0 - alpha / 2.0, wins + 1.0, n - wins),
                      1.0)

    lo = np.where(n > 0, np.nan_to_num(lo, nan=0.0), 0.0)
    hi = np.where(n > 0, np.nan_to_num(hi, nan=1.0), 1.0)
    return lo, hi


def _interval(wins, n):
    """Dispatch to the interval method chosen in the config."""
    if ci_method == "wilson":
        return wilson_interval(wins, n, ci_alpha)
    if ci_method == "clopper_pearson":
        return clopper_pearson_interval(wins, n, ci_alpha)
    raise ValueError(f"Unknown ci_method: {ci_method!r}")

In [9]:

@dataclass
class SufficiencyReport:
    subset: Tuple[str, ...]
    passes: bool                    # median width <= cfg.max_median_ci_width ?
    median_ci_width: float          # THE rejection statistic
    traffic_weighted_width: float   # extra diagnostic (weights = cell size)
    n_contexts: int                 # how many contexts the subset induces
    n_cells: int                    # contexts x buckets actually scored
    frac_empty_cells: float         # coverage gaps in the full grid
    cell_table: pd.DataFrame        # per-cell n / wins / rate / CI


def sufficiency_check(y: pd.Series, ctx: pd.Series, bid_bucket: pd.Series, subset: Tuple[str, ...]) -> SufficiencyReport:
    cells = pd.DataFrame({
        "ctx": ctx.to_numpy(),
        "bucket": bid_bucket.to_numpy(),   # Interval objects; groupable
        "y": y.to_numpy(),
    })

    # Per-cell counts: n = impressions in the cell, wins = won auctions.
    agg = (cells.groupby(["ctx", "bucket"], observed=True)["y"]
                .agg(n="size", wins="sum"))

    # Optionally expand to the FULL grid (every context x every bucket) so
    # that bid ranges a context never sees count as width-1.0 cells.
    full_grid = pd.MultiIndex.from_product(
        [np.unique(cells["ctx"]), list(bid_bucket.cat.categories)],
        names=["ctx", "bucket"],
    )
    agg = agg.reindex(full_grid, fill_value=0)

    n = agg["n"].to_numpy(dtype=float)
    wins = agg["wins"].to_numpy(dtype=float)

    lo, hi = _interval(wins, n)
    width = hi - lo                                  # empty cells -> 1.0

    median_width = float(np.median(width))
    # Traffic-weighted mean width: "how uncertain is the curve for the
    # average impression?" (empty cells naturally drop out, weight 0).
    populated = n > 0
    weighted_width = (float(np.average(width[populated], weights=n[populated])) if populated.any() else 1.0)

    table = agg.reset_index()
    table["win_rate"] = np.where(n > 0, wins / np.maximum(n, 1), np.nan)
    table["ci_lo"], table["ci_hi"], table["ci_width"] = lo, hi, width

    return SufficiencyReport(
        subset=subset,
        passes=median_width <= max_median_ci_width,
        median_ci_width=median_width,
        traffic_weighted_width=weighted_width,
        n_contexts=int(cells["ctx"].nunique()),
        n_cells=int(len(agg)),
        frac_empty_cells=float(np.mean(n == 0)),
        cell_table=table,
    )

In [10]:
def make_context_key(binned: pd.DataFrame, subset: Tuple[str, ...]) -> pd.Series:
    if len(subset) == 0:
        return pd.Series("GLOBAL", index=binned.index)
    return binned[list(subset)].agg("|".join, axis=1)

In [11]:
def _fit_isotonic(bids: np.ndarray, y: np.ndarray) -> IsotonicRegression:
    return IsotonicRegression(y_min=0.0, y_max=1.0, increasing=True, out_of_bounds="clip").fit(bids, y)


@dataclass
class CVResult:
    subset: Tuple[str, ...]
    mean_log_loss: float
    std_log_loss: float
    fold_log_losses: List[float]
    # Share of validation rows scored by their OWN context model (the rest
    # fell back to the global curve).  Low values reveal fragmentation even
    # before the sufficiency gate does.
    frac_context_scored: float


def cv_log_loss_for_subset(df: pd.DataFrame, binned: pd.DataFrame, subset: Tuple[str, ...]) -> CVResult:
    y_all = df['bid_won'].to_numpy()
    bids = df['bid'].to_numpy(dtype=float)
    ctx = make_context_key(binned, subset).to_numpy()

    # NOTE (temporal data): bidding logs usually drift over time.  For a
    # production system replace StratifiedKFold with time-ordered splits,
    # e.g. sklearn.model_selection.TimeSeriesSplit, keeping the rest as-is.
    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=random_state)

    losses: List[float] = []
    ctx_scored: List[float] = []

    for train_idx, val_idx in skf.split(np.zeros(len(df)), y_all):
        # ---- fit: one global fallback + one model per big-enough context ----
        global_model = _fit_isotonic(bids[train_idx], y_all[train_idx])

        train_frame = pd.DataFrame({"ctx": ctx[train_idx],
                                    "bid": bids[train_idx],
                                    "y": y_all[train_idx]})
        models: Dict[str, IsotonicRegression] = {}
        for c, g in train_frame.groupby("ctx", sort=False):
            # A context earns its own curve only with enough rows AND at
            # least two distinct bid values (otherwise no curve to speak of).
            # Contexts with constant y still get a curve (a constant one) --
            # if that constant is overconfident, CV log-loss will punish it,
            # which is precisely the selection mechanism at work.
            if len(g) >= min_context_train_rows and g["bid"].nunique() >= 2:
                models[c] = _fit_isotonic(g["bid"].to_numpy(),
                                          g["y"].to_numpy())

        # ---- predict on the validation fold, context by context -------------
        preds = np.empty(len(val_idx), dtype=float)
        val_frame = pd.DataFrame({"ctx": ctx[val_idx], "bid": bids[val_idx]},
                                 index=np.arange(len(val_idx)))
        n_context_scored = 0
        for c, g in val_frame.groupby("ctx", sort=False):
            model = models.get(c)
            if model is None:                 # small or unseen context
                model = global_model
            else:
                n_context_scored += len(g)
            preds[g.index] = model.predict(g["bid"].to_numpy())

        # ---- score -----------------------------------------------------------
        preds = np.clip(preds, 1e-6, 1.0 - 1e-6)
        losses.append(log_loss(y_all[val_idx], preds, labels=[0, 1]))
        ctx_scored.append(n_context_scored / len(val_idx))

    return CVResult(subset=subset,
                    mean_log_loss=float(np.mean(losses)),
                    std_log_loss=float(np.std(losses)),
                    fold_log_losses=[float(l) for l in losses],
                    frac_context_scored=float(np.mean(ctx_scored)))

### context finding

In [17]:
def _fmt_subset(subset: Tuple[str, ...]) -> str:
    return "{" + ", ".join(subset) + "}" if subset else "{} (global)"

In [18]:
max_size = len(candidate_features)
base_subset: Tuple[str, ...] = ()
ctx: pd.Series = make_context_key(binned, base_subset)
base_suff = sufficiency_check(y, ctx, bid_buckets, base_subset)
base_cv = cv_log_loss_for_subset(df, binned, base_subset)

In [19]:
print(f"[step 0] baseline {_fmt_subset(base_subset)} : "
    f"CV log-loss = {base_cv.mean_log_loss:.5f}, "
    f"median CI width = {base_suff.median_ci_width:.3f}")

[step 0] baseline {} (global) : CV log-loss = 0.40003, median CI width = 0.021


In [20]:
best_subset, best_loss = base_subset, base_cv.mean_log_loss
remaining = list(candidate_features)
step = 0

In [21]:
while remaining and len(best_subset) < max_size:
    step += 1
    passing: List[Tuple[CVResult, SufficiencyReport]] = []

    for feat in remaining:
        candidate = best_subset + (feat,)
        ctx = make_context_key(binned, candidate)
        # (1) sufficiency gate -- reject before paying for CV.
        suff = sufficiency_check(y, ctx, bid_buckets, candidate)
        if not suff.passes:
            print(f"[step {step}] REJECT {_fmt_subset(candidate)} : "
                    f"median CI width {suff.median_ci_width:.3f} "
                    f"> {max_median_ci_width} "
                    f"({suff.n_contexts} contexts, "
                    f"{suff.frac_empty_cells:.0%} empty cells)")
            continue
        
        # (2) accuracy -- CV log-loss of the per-context isotonic model.
        cv = cv_log_loss_for_subset(df, binned, candidate)
        passing.append((cv, suff))
        print(f"[step {step}] eval   {_fmt_subset(candidate)} : "
                f"CV log-loss = {cv.mean_log_loss:.5f} "
                f"(+/-{cv.std_log_loss:.5f}), "
                f"median CI width = {suff.median_ci_width:.3f}, "
                f"context-scored rows = {cv.frac_context_scored:.0%}")

    if not passing:
        print(f"[step {step}] every remaining candidate failed the sufficiency gate -> stop.")
        break

    # Best passing candidate of this step.
    cv_best, suff_best = min(passing, key=lambda t: t[0].mean_log_loss)
    improvement = best_loss - cv_best.mean_log_loss

    if improvement > 0.002:
        best_subset = cv_best.subset
        best_loss = cv_best.mean_log_loss
        remaining.remove(best_subset[-1])         # consume the feature
        print(f"[step {step}] ACCEPT {_fmt_subset(best_subset)} (improvement {improvement:.5f})")
    else:
        print(f"[step {step}] best improvement {improvement:.5f} <= tolerance 0.002 -> stop.")
        break

best_subset=best_subset
best_log_loss=best_loss
baseline_log_loss=base_cv.mean_log_loss

[step 1] eval   {a} : CV log-loss = 0.29917 (+/-0.00417), median CI width = 0.028, context-scored rows = 100%
[step 1] eval   {b} : CV log-loss = 0.39073 (+/-0.00602), median CI width = 0.045, context-scored rows = 100%
[step 1] eval   {c} : CV log-loss = 0.40300 (+/-0.00632), median CI width = 0.043, context-scored rows = 100%
[step 1] eval   {d} : CV log-loss = 0.40338 (+/-0.00625), median CI width = 0.047, context-scored rows = 100%
[step 1] eval   {e} : CV log-loss = 0.40312 (+/-0.00583), median CI width = 0.047, context-scored rows = 100%
[step 1] eval   {f} : CV log-loss = 0.40103 (+/-0.00556), median CI width = 0.030, context-scored rows = 100%
[step 1] eval   {g} : CV log-loss = 0.40309 (+/-0.00548), median CI width = 0.048, context-scored rows = 100%
[step 1] ACCEPT {a} (improvement 0.10086)
[step 2] eval   {a, b} : CV log-loss = 0.29046 (+/-0.00899), median CI width = 0.057, context-scored rows = 100%
[step 2] eval   {a, c} : CV log-loss = 0.30524 (+/-0.00602), median CI widt

### win curve table

In [24]:
#binned = discretize_features(df, cfg)
ctx = make_context_key(binned, best_subset)
#buckets = assign_bid_buckets(df[cfg.bid_col], cfg)

frame = pd.DataFrame({"ctx": ctx.to_numpy(),
                        "bucket": bid_buckets.to_numpy(),
                        "bid": df['bid'].to_numpy(dtype=float),
                        "y": df['bid_won'].to_numpy()})

# ---- empirical cells (observed only; reporting, not gating) -------------
agg = (frame.groupby(["ctx", "bucket"], observed=True)
            .agg(n=("y", "size"), wins=("y", "sum"),
                    bid_median=("bid", "median"))
            .reset_index())

lo, hi = _interval(agg["wins"].to_numpy(float),
                    agg["n"].to_numpy(float))
agg["win_rate"] = agg["wins"] / agg["n"]
agg["ci_lo"], agg["ci_hi"] = lo, hi
agg["ci_width"] = agg["ci_hi"] - agg["ci_lo"]

# ---- isotonic curves fitted on ALL data (the production model) ----------
global_model = _fit_isotonic(frame["bid"].to_numpy(),
                                frame["y"].to_numpy())
models: Dict[str, IsotonicRegression] = {}
for c, g in frame.groupby("ctx", sort=False):
    if len(g) >= min_context_train_rows and g["bid"].nunique() >= 2:
        models[c] = _fit_isotonic(g["bid"].to_numpy(), g["y"].to_numpy())

# Evaluate each cell's curve at the cell's median bid.
iso = np.empty(len(agg), dtype=float)
for c, g in agg.groupby("ctx", sort=False):
    model = models.get(c, global_model)
    iso[g.index] = model.predict(g["bid_median"].to_numpy())
agg["iso_win_rate"] = iso

agg.sort_values(["ctx", "bid_median"]).reset_index(drop=True)

,ctx,bucket,n,wins,bid_median,win_rate,ci_lo,ci_hi,ci_width,iso_win_rate
0,"seg_high|(-0.00099651, 0.199]","(-0.0009595000000000001, 0.996]",159,1,0.439426,0.006289,0.001111,0.034761,0.033650,0.000000
1,"seg_high|(-0.00099651, 0.199]","(0.996, 1.978]",162,1,1.576895,0.006173,0.001090,0.034133,0.033042,0.006623
2,"seg_high|(-0.00099651, 0.199]","(1.978, 3.003]",165,2,2.484130,0.012121,0.003330,0.043112,0.039782,0.013216
3,"seg_high|(-0.00099651, 0.199]","(3.003, 4.02]",157,7,3.486770,0.044586,0.021763,0.089163,0.067400,0.056738
4,"seg_high|(-0.00099651, 0.199]","(4.02, 4.989]",148,14,4.506622,0.094595,0.057186,0.152516,0.095330,0.106383
...,...,...,...,...,...,...,...,...,...,...
145,"seg_mid|(0.797, 1.0]","(4.989, 6.014]",213,70,5.527700,0.328638,0.269081,0.394267,0.125186,0.343284
146,"seg_mid|(0.797, 1.0]","(6.014, 7.01]",229,121,6.490939,0.528384,0.463795,0.592037,0.128242,0.531250
147,"seg_mid|(0.797, 1.0]","(7.01, 8.024]",237,198,7.532136,0.835443,0.782961,0.877225,0.094264,0.902985
148,"seg_mid|(0.797, 1.0]","(8.024, 9.014]",236,225,8.510965,0.953390,0.918479,0.973777,0.055298,0.941691
